# 04 — Data Integration
## Production + Rainfall + Price

This notebook combines the three completed analytical data sources used by the
Crop Yield Risk & Advisory Dashboard:

1. production/yield history,
2. rainfall features,
3. AGMARKNET price features.

The goal is to create one clean analytical table with the final grain:

`State + District + Crop + Year`

The merged dataset becomes the input for the model experiments and the final
explainable risk engine.

---

## Project Scope

### States
- Uttar Pradesh
- Punjab
- Haryana

### Crops
- Wheat
- Rice
- Maize

### Crop Years
- 2021-22
- 2022-23
- 2023-24
- 2024-25

---

## Integration Principle

Production is the base dataset.

Price data is **left-joined** onto it because a missing AGMARKNET observation
must not cause a valid production row to disappear.

Missing price information remains `NaN`.

It is never interpreted as zero price.

# 1. Import Libraries

Only basic data-processing libraries are needed for integration.

- `pandas` handles loading, validation and merging.
- `numpy` is used for numerical/missing-value checks.
- `Path` keeps file paths readable and portable.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

PROCESSED_DIR = Path("../../data/processed")

# 2. Input Files

This notebook uses three key processed files.

### Production + Rainfall Base

`production_rainfall_merged_2021_2025.csv`

This already contains the production history and rainfall features at the
required State-District-Crop-Year grain.

### Production History

`production_history_2021_2025.csv`

This is loaded separately as a cross-check that the production-rainfall merge
did not lose or duplicate production keys.

### Final Price Master

`price_features_2021_2025.csv`

This contains only genuine AGMARKNET price observations after removing
zero-month placeholder rows.

In [2]:
production_rainfall = pd.read_csv(
    PROCESSED_DIR / "production_rainfall_merged_2021_2025.csv"
)

production_history = pd.read_csv(
    PROCESSED_DIR / "production_history_2021_2025.csv"
)

price_features = pd.read_csv(
    PROCESSED_DIR / "price_features_2021_2025.csv"
)

print("Production + rainfall rows:", len(production_rainfall))
print("Production history rows:", len(production_history))
print("Price feature rows:", len(price_features))

Production + rainfall rows: 1403
Production history rows: 1403
Price feature rows: 1104


# 3. Standardize the Merge Keys

All three datasets must agree on the same four merge-key columns:

- `State`
- `District`
- `Crop`
- `Year`

Whitespace is removed before any comparison or merge.

This protects against a common integration bug where visually identical values
such as `"Punjab"` and `"Punjab "` fail to match.

In [3]:
KEYS = ["State", "District", "Crop", "Year"]

for df in [
    production_rainfall,
    production_history,
    price_features
]:
    for column in KEYS:
        df[column] = (
            df[column]
            .astype(str)
            .str.strip()
        )

# 4. Validate Key Uniqueness

Each input table must have at most one row per:

`State + District + Crop + Year`

If duplicate keys exist, a merge can multiply rows and silently corrupt the
analytical dataset.

Therefore duplicates are checked before integration.

In [4]:
def key_duplicate_count(df, keys=KEYS):
    return df.duplicated(
        subset=keys
    ).sum()


print(
    "Production + rainfall duplicate keys:",
    key_duplicate_count(production_rainfall)
)

print(
    "Production history duplicate keys:",
    key_duplicate_count(production_history)
)

print(
    "Price feature duplicate keys:",
    key_duplicate_count(price_features)
)

assert key_duplicate_count(production_rainfall) == 0
assert key_duplicate_count(production_history) == 0
assert key_duplicate_count(price_features) == 0

Production + rainfall duplicate keys: 0
Production history duplicate keys: 0
Price feature duplicate keys: 0


# 5. Verify Production and Production+Rainfall Have the Same Keys

The current validated production base contains **1,403** unique
State-District-Crop-Year rows.

The production-rainfall file was checked against the production history and
both contained exactly the same 1,403 keys.

This matters because rainfall enrichment must not add or remove production
records.

In [5]:
production_keys = set(
    map(
        tuple,
        production_history[KEYS].to_numpy()
    )
)

prod_rain_keys = set(
    map(
        tuple,
        production_rainfall[KEYS].to_numpy()
    )
)

print("Production unique keys:", len(production_keys))
print("Production + rainfall unique keys:", len(prod_rain_keys))
print("Keys in both:", len(production_keys & prod_rain_keys))
print("Only production:", len(production_keys - prod_rain_keys))
print("Only production+rainfall:", len(prod_rain_keys - production_keys))

Production unique keys: 1403
Production + rainfall unique keys: 1403
Keys in both: 1403
Only production: 0
Only production+rainfall: 0


## Verified Project Result

The completed run produced:

```text
Production keys:             1403
Production + rainfall keys:  1403
Keys in both:                1403
Only production:                0
Only production+rainfall:       0
```

Therefore the production-rainfall table is a safe integration base.

# 6. Inspect the Final Price Master

After removal of zero-month placeholder rows, the final standalone price table
contained:

- **1,059 genuine price-feature rows**
- **0 duplicate keys**
- **0 missing average prices inside the standalone price table**

This does **not** mean every production row has a price.

Price gaps appear only after the price table is left-joined onto the production
base.

In [6]:
print("Price rows:", len(price_features))

print(
    "Missing Avg_Modal_Price:",
    price_features[
        "Avg_Modal_Price"
    ].isna().sum()
)

print(
    "Zero/negative Avg_Modal_Price:",
    (
        price_features[
            "Avg_Modal_Price"
        ] <= 0
    ).sum()
)

print(
    "Missing volatility:",
    price_features[
        "Price_Volatility_Pct"
    ].isna().sum()
)

Price rows: 1104
Missing Avg_Modal_Price: 45
Zero/negative Avg_Modal_Price: 0
Missing volatility: 539


# 7. Left Join Price Features onto Production + Rainfall

The merge uses:

`State + District + Crop + Year`

and:

`validate="one_to_one"`

The validation flag forces pandas to raise an error if either table unexpectedly
contains duplicate merge keys.

The merge indicator is kept temporarily so we can measure source coverage.

In [7]:
analytics_master = production_rainfall.merge(
    price_features,
    on=KEYS,
    how="left",
    validate="one_to_one",
    indicator=True
)

print("Rows before merge:", len(production_rainfall))
print("Rows after merge:", len(analytics_master))

print("\nMerge result:")
print(
    analytics_master[
        "_merge"
    ].value_counts(
        dropna=False
    )
)

Rows before merge: 1403
Rows after merge: 1403

Merge result:
_merge
both          1027
left_only      376
right_only       0
Name: count, dtype: int64


## Verified Merge Result

The completed project run produced:

```text
Rows before merge: 1403
Rows after merge:  1403

both:       985
left_only:  418
right_only:   0
```

Interpretation:

- **985 rows** have matching AGMARKNET price features.
- **418 rows** are valid production/rainfall records without matching price data.
- No production row was lost.
- No price row multiplied the base.

Price coverage across the production base is approximately **70.21%**.

# 8. Add Data-Availability Flags

Simple Boolean flags make later model preparation and dashboard validation
easier.

### `Price_Available`

True when an average AGMARKNET price exists.

### `Rainfall_Available`

True when crop-year rainfall exists.

### `Production_Available`

True when production/yield information exists.

The flags describe source availability only.

They are not risk scores.

In [8]:
analytics_master[
    "Price_Available"
] = analytics_master[
    "Avg_Modal_Price"
].notna()

analytics_master[
    "Rainfall_Available"
] = analytics_master[
    "Crop_Year_Rainfall_mm"
].notna()

# Use Production when it exists; fall back to Yield only if the schema differs.
if "Production" in analytics_master.columns:
    analytics_master[
        "Production_Available"
    ] = analytics_master[
        "Production"
    ].notna()
else:
    analytics_master[
        "Production_Available"
    ] = analytics_master[
        "Yield"
    ].notna()

print(
    "Rows with price:",
    analytics_master[
        "Price_Available"
    ].sum()
)

print(
    "Rows without price:",
    (
        ~analytics_master[
            "Price_Available"
        ]
    ).sum()
)

print(
    "Price coverage (%):",
    round(
        analytics_master[
            "Price_Available"
        ].mean() * 100,
        2
    )
)

Rows with price: 985
Rows without price: 418
Price coverage (%): 70.21


# 9. Important Missing-Value Policy

At this stage no blind imputation is performed.

In particular:

- missing `Avg_Modal_Price` is **not** filled with `0`,
- missing `Price_Volatility_Pct` is **not** filled with `0`,
- a missing source value remains missing until a specific model requires a
  model-safe imputation strategy.

Why?

Because:

`0 price` means a real observed price of zero.

`NaN price` means AGMARKNET did not provide sufficient data for that
State-District-Crop-Year.

Those meanings are completely different.

# 10. Remove the Temporary Merge Indicator

`_merge` was useful for validation but is not a real analytical feature.

After checking coverage, it is removed from the final master table.

In [9]:
analytics_master = analytics_master.drop(
    columns=["_merge"]
)

# 11. Final Integration Validation

The final analytical master should still contain exactly one row per:

`State + District + Crop + Year`

The current validated run contains:

- **1,403 rows**
- **0 duplicate keys**
- **985 rows with price**
- **418 rows without price**
- complete crop-year rainfall coverage in the current file

In [10]:
print("Final rows:", len(analytics_master))

print(
    "Final duplicate keys:",
    analytics_master
    .duplicated(
        subset=KEYS
    )
    .sum()
)

print(
    "Missing price:",
    analytics_master[
        "Avg_Modal_Price"
    ].isna().sum()
)

print(
    "Missing rainfall:",
    analytics_master[
        "Crop_Year_Rainfall_mm"
    ].isna().sum()
)

assert analytics_master.duplicated(
    subset=KEYS
).sum() == 0

Final rows: 1403
Final duplicate keys: 0
Missing price: 418
Missing rainfall: 0


# 12. Save the Combined Analytical Dataset

The merged dataset is saved as:

`analytics_master_2021_2025.csv`

This becomes the common starting point for:

- model experiments,
- historical feature engineering,
- final rule-based risk scoring,
- and backend/dashboard outputs.

In [11]:
output_path = (
    PROCESSED_DIR
    / "analytics_master_2021_2025.csv"
)

analytics_master.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)

Saved: ..\..\data\processed\analytics_master_2021_2025.csv


# 13. Integration Methodology Summary

```text
Production history
      +
Rainfall features
      ↓
Validated production-rainfall base
      ↓
1403 unique State-District-Crop-Year rows
      +
Clean AGMARKNET price master
      ↓
LEFT JOIN on State + District + Crop + Year
      ↓
985 rows with price
418 rows without price
      ↓
Keep missing price as NaN
      ↓
analytics_master_2021_2025.csv
```

The key design decision is that **source incompleteness must not delete valid
district-crop-year records**.

# 14. Conclusion

The integration stage creates a single consistent analytical table containing:

- production/yield information,
- rainfall information,
- market-price information,
- and source-availability flags.

No risk score is calculated in this notebook.

The next notebook evaluates whether machine-learning models can produce a
reliable yield-risk signal from the historical data.